In [1]:
%pip install highspy highspy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Импорты
from pathlib import Path
import numpy as np
import highspy

##### Для игр с нулевой суммой:

$A = (a_{ij})$, где:
- строки — стратегии игрока **X** (максимизирует выигрыш),
- столбцы — стратегии игрока **Y** (минимизирует выигрыш).

**Верхняя цена игры (minimax):**

$
\overline{v} = \min_{x} \; \max_{y} \; a_{ij}
$

**Нижняя цена игры (maximin):**

$
\underline{v} = \max_{y} \; \min_{x} \; a_{ij}
$

In [6]:
def read_matrix(filepath):
    A = np.loadtxt(filepath, dtype=np.double, delimiter=";")
    return A


def find_value_bounds(A):
    """Maximin (lower value) & minimax (upper value)"""
    row_mins = A.min(axis=1)
    col_maxs = A.max(axis=0)
    return row_mins.max(), col_maxs.min()


def normalize(A):
    shift = -A.min() + 1 if A.min() <= 0 else 0
    return A + shift, shift


def solve_game(A):
    m, n = A.shape
    highs = highspy.Highs()
    inf = highspy.kHighsInf

    # Variables x_i >= 0
    for i in range(m):
        highs.addVar(0.0, inf)

    # Constraints A^T @ x >= 1 (for player X)
    for j in range(n):
        indices = np.arange(m)
        values = np.array(A[:, j])
        highs.addRow(1.0, inf, m, indices, values)

    # Objective function
    for i in range(m):
        highs.changeColCost(i, 1.0)

    highs.run()
    sol = highs.getSolution()
    info = highs.getInfo()

    v = 1 / info.objective_function_value
    p = np.array(sol.col_value) * v
    q = np.array(sol.row_dual) * v

    return v, p, q


def validate(lv, uv, A, p, q, v):
    assert lv <= v <= uv, f"Game value {v} is outside allowed range [{lv}, {uv}]"
    assert np.isclose(p.sum(), 1.0), (
        f"Incorrect player X possibilities {p}, sum is: {p.sum()}"
    )
    assert np.isclose(q.sum(), 1.0), (
        f"Incorrect player Y possibilities {q}, sum is: {q.sum()}"
    )
    assert np.all(p @ A >= v - 1e-8), f"Player X strategy is not optimal: {p @ A}"
    assert np.all(A @ q <= v + 1e-8), f"Player Y strategy is not optimal: {A @ q}"


def run(filepath):
    A = read_matrix(filepath)
    lv, uv = find_value_bounds(A)
    print(filepath)
    print("-----------------------------")
    print(f"Lower value: {lv}\nUpper value: {uv}")
    A, shift = normalize(A)
    print(f"Shift: {shift}")
    v, p, q = solve_game(A)
    v -= shift
    A -= shift
    print("Value:", v)
    print("p:", p)
    print("q:", q)
    print("-----------------------------")
    validate(lv, uv, A, p, q, v)

In [ ]:
np.set_printoptions(precision=2, suppress=True)

print("-----------------------------")
inputs = Path("inputs")
for file in inputs.iterdir():
    run(file)

-----------------------------
inputs/A_1mln_100.csv
-----------------------------
Lower value: -708.0
Upper value: 999.0
Shift: 1001.0


MemoryError: bad allocation